# IIT-7. Intégration par coupes — la coupe et le verdict (Lean)

## Navigation
**Série** : [IIT](./README.md) | **Partie** : Escalier Lean (marche 1/3)
**Précédent** : [IIT-6 : L'objet qui a mordu IIT](./IIT-06-L-Objet-qui-a-Mordu-IIT-Python.ipynb) | **Suivant** : --

## Position dans l'escalier

Ce carnet est la **première marche** d'un escalier de trois qui expose le lake Lean
[`iit_lean`](../SymbolicAI/Lean/iit_lean/README.md) — la moitié **formelle** de la veine
« l'objet qui a mordu IIT » de l'Epic Aaronson ([#16781](https://github.com/jsboige/CoursIA/issues/16781)).
La moitié **empirique** (Φ_DM sur F_p, grilles PyPhi) est mesurée dans
[IIT-6](./IIT-06-L-Objet-qui-a-Mordu-IIT-Python.ipynb) ; les deux moitiés se rejoignent à la marche 3.

| Marche | Contenu | État |
|---|---|---|
| **1 — la coupe et le verdict** (ce carnet) | vocabulaire (`State`, `System`, `Cut`), motifs, `cutIndep`, verdict `decomposable`/`integrated` instancié | livré |
| 2 — la parité, de bout en bout | `parity`, `broadcast`, les quatre lemmes de parité calculés | à venir |
| 3 — l'objet qui a mordu IIT, en théorèmes | `broadcast_integrated` + `broadcast_two_cycle` : intégration universelle, dynamique triviale | à venir |

## Objectifs d'apprentissage

1. **Tenir** un système déterministe fini comme un objet Lean (`System n`), une coupe (`Cut n`) et ses motifs (`projT`/`projF`)
2. **Lire** la définition relationnelle de l'indépendance d'une coupe (`cutIndep`) comme un produit d'images
3. **Faire rendre un verdict** : prouver `decomposable` sur une instance en invoquant `product_decomposable`, et voir la même machinerie **buter** sur le broadcast-parité
4. **Situer honnêtement** ce verdict : un proxy relationnel, pas Φ — ce qu'il ne dit pas

**Prérequis** : logique booléenne ; la lecture d'un `#check` suffit, aucune expérience Lean
n'est exigée au-delà de [Lean-3](../SymbolicAI/Lean/Lean-03-Propositions-Proofs-Lean.ipynb).
Kernel `lean4-wsl`, toolchain `leanprover/lean4:v4.33.0`, sans Mathlib : le lake se compile en secondes.


## 1. Environnement — charger le lake

L'import ci-dessous charge le module `IIT.Integration` du lake (compilé en `.olean` par
`lake build`). Le contrôle positif `#eval 2 + 2` vérifie que le kernel est vivant et que sa
toolchain est cohérente avec celle du lake : si cette cellule échoue, rien de ce qui suit n'a de valeur.


In [1]:
import IIT.Integration
open IIT

-- Controle positif : le kernel doit rendre 4.
#eval 2 + 2

-- Le lake est charge : les definitions de la marche 1 sont visibles.
#check @IIT.State
#check @IIT.System
#check @IIT.Cut


import IIT.Integration
open IIT

-- Controle positif : le kernel doit rendre 4.
#eval 2 + 2
─────▶  4

-- Le lake est charge : les definitions de la marche 1 sont visibles.
#check @IIT.State
──────▶  State : Nat → Type
#check @IIT.System
──────▶  System : Nat → Type
#check @IIT.Cut
──────▶  Cut : Nat → Type

--% env 0

Raw input:
{"cmd": "import IIT.Integration\nopen IIT\n\n-- Controle positif : le kernel doit rendre 4.\n#eval 2 + 2\n\n-- Le lake est charge : les definitions de la marche 1 sont visibles.\n#check @IIT.State\n#check @IIT.System\n#check @IIT.Cut\n"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "4"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data": "State : Nat → Type"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data": "System : Nat → Type"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data": "Cut : Nat → Type"}],
 "env": 0}

**Interprétation.** `4`, puis les trois signatures. Le kernel tourne sur la toolchain du lake
(`v4.33.0`) et voit ses déclarations : nous sommes dans le lake, pas sur un stub. Tout ce que ce
carnet exécutera se réduit à du core Lean (`Fin`, `Bool`, `xor`) — le lake n'a **aucune** dépendance
Mathlib, c'est pourquoi il se compile en secondes et se prête à l'expérimentation interactive.


## 2. Un système est une fonction — `State`, `System`

Le lake formalise les systèmes déterministes à `n` éléments booléens :

- un **état** est une valuation booléenne des `n` éléments : `State n := Fin n → Bool` ;
- un **système** est une fonction d'états — la matrice de transition, mais **sans** probabilities :
  `System n := State n → State n`.

C'est le dual déterministe des TPM de PyPhi ([IIT-1](./IIT-01-IntroToPyPhi-Python.ipynb)) : ici, un état
détermine exactement le suivant. Construisons un état et un système concrets sur `n = 3`.


In [2]:
-- Un etat concret de n = 3 : seul l'element 0 est vrai.
def etatOneHot : State 3 := fun i => i.val == 0

-- Un systeme concret : chaque element recopie son voisin de droite (cycliquement).
def decalage : System 3 := fun s i => s ((i + 1) % 3)

-- L'etat et son image, composante par composante.
#eval [etatOneHot 0, etatOneHot 1, etatOneHot 2]
#eval [decalage etatOneHot 0, decalage etatOneHot 1, decalage etatOneHot 2]

-- Une deuxieme iteration confirme le determinisme : meme entree, meme sortie.
#eval [decalage (decalage etatOneHot) 0, decalage (decalage etatOneHot) 1, decalage (decalage etatOneHot) 2]


-- Un etat concret de n = 3 : seul l'element 0 est vrai.
def etatOneHot : State 3 := fun i => i.val == 0

-- Un systeme concret : chaque element recopie son voisin de droite (cycliquement).
def decalage : System 3 := fun s i => s ((i + 1) % 3)

-- L'etat et son image, composante par composante.
#eval [etatOneHot 0, etatOneHot 1, etatOneHot 2]
─────▶  [true, false, false]
#eval [decalage etatOneHot 0, decalage etatOneHot 1, decalage etatOneHot 2]
─────▶  [false, false, true]

-- Une deuxieme iteration confirme le determinisme : meme entree, meme sortie.
#eval [decalage (decalage etatOneHot) 0, decalage (decalage etatOneHot) 1, decalage (decalage etatOneHot) 2]
─────▶  [false, true, false]

--% env 1

Raw input:
{"cmd": "-- Un etat concret de n = 3 : seul l'element 0 est vrai.\ndef etatOneHot : State 3 := fun i => i.val == 0\n\n-- Un systeme concret : chaque element recopie son voisin de droite (cycliquement).\ndef decalage : System 3 := fun s i => s ((i + 1) % 3)\n\n-- L'etat et son image, composante par composante.\n#eval [etatOneHot 0, etatOneHot 1, etatOneHot 2]\n#eval [decalage etatOneHot 0, decalage etatOneHot 1, decalage etatOneHot 2]\n\n-- Une deuxieme iteration confirme le determinisme : meme entree, meme sortie.\n#eval [decalage (decalage etatOneHot) 0, decalage (decalage etatOneHot) 1, decalage (decalage etatOneHot) 2]\n", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "[true, false, false]"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 5},
   "data": "[false, false, true]"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 5},
   "data": "[false, true, false]"}],
 "env": 1}

**Interprétation.** L'état `[true, false, false]` devient `[false, false, true]` : l'unique bit vrai
a été décalé d'un cran. La deuxième itération (la composée `decalage (decalage etatOneHot)` du
`#eval` suivant) rend `[false, true, false]` — même motif, un cran plus loin. Une fonction d'états est un objet manipulable comme n'importe quelle
fonction Lean — on la **compose** (`decalage (decalage etatOneHot)`), on l'évalue composante par
composante. Aucune énumération de matrice n'est nécessaire : la fonction *est* la table.


## 3. Couper — `Cut` et les motifs `projT`/`projF`

Une **coupe** partage les `n` éléments en deux côtés **non vides** : le champ `sel` place chaque
élément côté A (`true`) ou côté B (`false`), et les champs `ha`/`hb` attestent que chaque côté
existe. Cette non-trivialité n'est pas décorative : une coupe au côté vide rendrait l'indépendance
triviale — tout système serait « décomposable » au sens vide du terme.

Les **motifs** d'un état pour une coupe sont ses deux moitiés masquées : `projT` préserve les bits
côté A et met le côté B à `false`, `projF` fait le miroir. Ce sont les rouages de `cutIndep` —
voyons-les sur un état concret **avant** de lire la définition.


In [3]:
-- Une coupe de n = 3 : A = {0, 1}, B = {2}.
def coupeAB : Cut 3 where
  sel := fun i => decide (i.val < 2)
  ha := ⟨0, rfl⟩
  hb := ⟨2, rfl⟩

-- Un etat quelconque.
def etatMelange : State 3 := fun i => i.val ≠ 1

#eval [etatMelange 0, etatMelange 1, etatMelange 2]
#eval [projT coupeAB etatMelange 0, projT coupeAB etatMelange 1, projT coupeAB etatMelange 2]
#eval [projF coupeAB etatMelange 0, projF coupeAB etatMelange 1, projF coupeAB etatMelange 2]


-- Une coupe de n = 3 : A = {0, 1}, B = {2}.
def coupeAB : Cut 3 where
  sel := fun i => decide (i.val < 2)
  ha := ⟨0, rfl⟩
  hb := ⟨2, rfl⟩

-- Un etat quelconque.
def etatMelange : State 3 := fun i => i.val ≠ 1

#eval [etatMelange 0, etatMelange 1, etatMelange 2]
─────▶  [true, false, true]
#eval [projT coupeAB etatMelange 0, projT coupeAB etatMelange 1, projT coupeAB etatMelange 2]
─────▶  [true, false, false]
#eval [projF coupeAB etatMelange 0, projF coupeAB etatMelange 1, projF coupeAB etatMelange 2]
─────▶  [false, false, true]

--% env 2

Raw input:
{"cmd": "-- Une coupe de n = 3 : A = {0, 1}, B = {2}.\ndef coupeAB : Cut 3 where\n  sel := fun i => decide (i.val < 2)\n  ha := \u27e80, rfl\u27e9\n  hb := \u27e82, rfl\u27e9\n\n-- Un etat quelconque.\ndef etatMelange : State 3 := fun i => i.val \u2260 1\n\n#eval [etatMelange 0, etatMelange 1, etatMelange 2]\n#eval [projT coupeAB etatMelange 0, projT coupeAB etatMelange 1, projT coupeAB etatMelange 2]\n#eval [projF coupeAB etatMelange 0, projF coupeAB etatMelange 1, projF coupeAB etatMelange 2]\n", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 5},
   "data": "[true, false, true]"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "[true, false, false]"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 5},
   "data": "[false, false, true]"}],
 "env": 2}

**Interprétation.** L'état `[true, false, true]` se coupe en motif A `[true, false, false]`
(le bit 1 est côté A mais faux, le bit 2 est masqué) et motif B `[false, false, true]` (les bits
côté A sont masqués, le bit 2 passe). Les hypothèses `ha` et `hb` sont closes par `rfl` : sur des
littéraux, `decide (0 < 2)` et `decide (2 < 2)` réduisent par calcul — la non-trivialité de la
coupe est un fait **calculé**, pas admis.


## 4. Le verdict — `cutIndep`, `decomposable`, `integrated`

La définition centrale du lake est **relationnelle** :

```lean
def cutIndep (c : Cut n) (f : System n) : Prop :=
  ∀ x y : State n, ∃ z : State n,
    projT c (f z) = projT c (f x) ∧ projF c (f z) = projF c (f y)
```

Lue à voix haute : *quels que soient deux états `x` et `y`, il existe un état `z` dont l'image
réalise le motif A de `f x` **et** le motif B de `f y` simultanément.* Autrement dit, l'image du
système est **exactement le produit des images de ses deux parties** — rien dans la sortie ne mêle
les côtés. Le verdict s'en dérive en une ligne chacun :


In [4]:
#check @IIT.cutIndep
#check @IIT.decomposable
#check @IIT.integrated

-- decomposable : ∃ une coupe independante ; integrated : ¬ decomposable.
#print IIT.decomposable
#print IIT.integrated


#check @IIT.cutIndep
──────▶  @cutIndep : {n : Nat} → Cut n → System n → Prop
#check @IIT.decomposable
──────▶  @decomposable : {n : Nat} → System n → Prop
#check @IIT.integrated
──────▶  @integrated : {n : Nat} → System n → Prop

-- decomposable : ∃ une coupe independante ; integrated : ¬ decomposable.
#print IIT.decomposable
──────▶  def IIT.decomposable : {n : Nat} → System n → Prop :=
fun {n} f => ∃ c, cutIndep c f
#print IIT.integrated
──────▶  def IIT.integrated : {n : Nat} → System n → Prop :=
fun {n} f => ¬decomposable f

--% env 3

Raw input:
{"cmd": "#check @IIT.cutIndep\n#check @IIT.decomposable\n#check @IIT.integrated\n\n-- decomposable : \u2203 une coupe independante ; integrated : \u00ac decomposable.\n#print IIT.decomposable\n#print IIT.integrated\n", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "@cutIndep : {n : Nat} → Cut n → System n → Prop"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "@decomposable : {n : Nat} → System n → Prop"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "@integrated : {n : Nat} → System n → Prop"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "def IIT.decomposable : {n : Nat} → System n → Prop :=\nfun {n} f => ∃ c, cutIndep c f"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "def IIT.integrated : {n : Nat} → System n → Prop :=\nfun {n} f => ¬decomposable f"}],
 "env": 3}

**Interprétation.** `decomposable f` dit *il existe une coupe qui sépare `f`* ; `integrated f`
dit *aucune coupe ne le sépare*. Deux points de lecture essentiels :

1. **C'est un verdict, pas une mesure.** Pas de nombre, pas de degré : un système est décomposable
   ou il ne l'est pas. Φ — la quantité de la théorie — est ailleurs (voir le grade en §7).
2. **L'ordre des quantificateurs fait la force.** `cutIndep` exige le couple `(x, y)` **pour
   tous** les choix — le système doit pouvoir mélanger *n'importe quelle* paire de motifs
   réalisables séparément. C'est ce « pour tout » que le broadcast violera en §6.


## 5. Théorème A — les systèmes-produits sont décomposables

Le premier théorème du lake transporte une hypothèse **locale** (chaque côté de la coupe n'influence
que lui-même) en une conclusion **globale** (la coupe est indépendante) :

```lean
theorem cutIndep_of_sides_independent (c : Cut n) (f : System n)
    (hA : ∀ s s', (∀ i, c.sel i = true → s i = s' i) → ∀ i, c.sel i = true → f s i = f s' i)
    (hB : ∀ s s', (∀ i, c.sel i = false → s i = s' i) → ∀ i, c.sel i = false → f s i = f s' i)
```

`hA` se lit : *deux états qui coïncident côté A ont des images qui coïncident côté A* — le côté A
de la sortie ne dépend que du côté A de l'entrée. `hB` est le miroir côté B. La preuve du lake est
un **témoin constructif** : le mélange `fun i => if c.sel i then x i else y i` réalise tout couple
de motifs. Vérifions les signatures, puis **instancions** le théorème sur un système que nous
construisons nous-mêmes.


In [5]:
#check @IIT.cutIndep_of_sides_independent
#check @IIT.product_decomposable

-- Ce que "prouve" veut dire : propext et Quot.sound (les deux axiomes du core),
-- ni Classical.choice ni sorryAx.
#print axioms IIT.product_decomposable


#check @IIT.cutIndep_of_sides_independent
──────▶  @cutIndep_of_sides_independent : ∀ {n : Nat} (c : Cut n) (f : System n),
  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →
    (∀ (s s' : State n),
        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →
      cutIndep c f
#check @IIT.product_decomposable
──────▶  @product_decomposable : ∀ {n : Nat} (c : Cut n) (f : System n),
  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →
    (∀ (s s' : State n),
        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →
      decomposable f

-- Ce que "prouve" veut dire : propext et Quot.sound (les deux axiomes du core),
-- ni Classical.choice ni sorryAx.
#print axioms IIT.product_decomposable
──────▶  'IIT.product_decomposable' depends on axioms: [propext, Quot.sound]

--% env 4

Raw input:
{"cmd": "#check @IIT.cutIndep_of_sides_independent\n#check @IIT.product_decomposable\n\n-- Ce que \"prouve\" veut dire : propext et Quot.sound (les deux axiomes du core),\n-- ni Classical.choice ni sorryAx.\n#print axioms IIT.product_decomposable\n", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "@cutIndep_of_sides_independent : ∀ {n : Nat} (c : Cut n) (f : System n),\n  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →\n    (∀ (s s' : State n),\n        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →\n      cutIndep c f"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@product_decomposable : ∀ {n : Nat} (c : Cut n) (f : System n),\n  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →\n    (∀ (s s' : State n),\n        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →\n      decomposable f"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "'IIT.product_decomposable' depends on axioms: [propext, Quot.sound]"}],
 "env": 4}

**Interprétation.** `product_decomposable` est l'alias *figé* du théorème (le nom sous lequel
l'issue [#18599](https://github.com/jsboige/CoursIA/issues/18599) l'a commandé) : mêmes
hypothèses, conclusion `decomposable f`. Et `#print axioms` répond *« depends on axioms: [propext, Quot.sound] »* — les deux
axiomes du cœur de Lean (`propext`, `Quot.sound`), ni `Classical.choice` ni `sorryAx` : la preuve
reste constructive jusqu'au bout, sans `sorry` caché ni choix non-constructif. C'est le grade
attendu d'une preuve du core : les deux axiomes structurels, jamais les axiomes de triche.


### L'instance : deux moitiés autonomes, dont une qui s'active en interne

Construisons sur `n = 4` un système où le côté A `{0, 1}` vit sa vie (recopie ses bits) et le côté
B `{2, 3}` vit la sienne — mais pas inertement : **il échange ses deux bits**. Chaque côté est
autonome (il ne lit que lui-même), et pourtant aucun des deux n'est trivial.


In [6]:
-- Coupe de n = 4 : A = {0, 1}, B = {2, 3}.
def cutAB4 : Cut 4 where
  sel := fun i => decide (i.val < 2)
  ha := ⟨0, rfl⟩
  hb := ⟨2, rfl⟩

-- Cote A : recopie. Cote B : echange ses deux bits.
def echange : System 4 := fun s i =>
  if i.val = 2 then s 3 else if i.val = 3 then s 2 else s i

-- Un etat temoin : [true, false, false, true] -- le cote B porte un bit vrai en 3.
def etatTemoin : State 4 := fun i => i.val == 0 || i.val == 3

#eval [etatTemoin 0, etatTemoin 1, etatTemoin 2, etatTemoin 3]
#eval [echange etatTemoin 0, echange etatTemoin 1, echange etatTemoin 2, echange etatTemoin 3]


-- Coupe de n = 4 : A = {0, 1}, B = {2, 3}.
def cutAB4 : Cut 4 where
  sel := fun i => decide (i.val < 2)
  ha := ⟨0, rfl⟩
  hb := ⟨2, rfl⟩

-- Cote A : recopie. Cote B : echange ses deux bits.
def echange : System 4 := fun s i =>
  if i.val = 2 then s 3 else if i.val = 3 then s 2 else s i

-- Un etat temoin : [true, false, false, true] -- le cote B porte un bit vrai en 3.
def etatTemoin : State 4 := fun i => i.val == 0 || i.val == 3

#eval [etatTemoin 0, etatTemoin 1, etatTemoin 2, etatTemoin 3]
─────▶  [true, false, false, true]
#eval [echange etatTemoin 0, echange etatTemoin 1, echange etatTemoin 2, echange etatTemoin 3]
─────▶  [true, false, true, false]

--% env 5

Raw input:
{"cmd": "-- Coupe de n = 4 : A = {0, 1}, B = {2, 3}.\ndef cutAB4 : Cut 4 where\n  sel := fun i => decide (i.val < 2)\n  ha := \u27e80, rfl\u27e9\n  hb := \u27e82, rfl\u27e9\n\n-- Cote A : recopie. Cote B : echange ses deux bits.\ndef echange : System 4 := fun s i =>\n  if i.val = 2 then s 3 else if i.val = 3 then s 2 else s i\n\n-- Un etat temoin : [true, false, false, true] -- le cote B porte un bit vrai en 3.\ndef etatTemoin : State 4 := fun i => i.val == 0 || i.val == 3\n\n#eval [etatTemoin 0, etatTemoin 1, etatTemoin 2, etatTemoin 3]\n#eval [echange etatTemoin 0, echange etatTemoin 1, echange etatTemoin 2, echange etatTemoin 3]\n", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "[true, false, false, true]"},
  {"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 5},
   "data": "[true, false, true, false]"}],
 "env": 5}

**Interprétation.** L'image de `[true, false, false, true]` est `[true, false, true, false]` :
le côté A `[true, false]` est intact, le côté B `[false, true]` est devenu `[true, false]` —
l'échange interne a agi. Le système n'est **pas** l'identité : il fait quelque chose. La question
de la marche : quel **verdict** rend `cutIndep` sur lui ? Le théorème A dit *décomposable* —
prouvons-le en l'invoquant, hypothèses comprises.


In [7]:
-- Le verdict, par le theoreme A : chaque cote ne lit que lui-meme.
theorem echange_decomposable : decomposable echange := by
  refine product_decomposable cutAB4 echange ?_ ?_
  · -- Cote A : deux etats qui coïncident sur {0,1} ont des images qui y coïncident.
    intro s s' heq i hi
    have hi2 : i.val < 2 := by simpa [cutAB4] using hi
    by_cases h2 : i.val = 2
    · exact absurd h2 (by omega)
    · have h3 : i.val ≠ 3 := by omega
      simp only [echange, if_neg h2, if_neg h3]
      exact heq i hi
  · -- Cote B : deux etats qui coïncident sur {2,3} ont des images qui y coïncident.
    intro s s' heq i hi
    have hlt : ¬ (i.val < 2) := by simpa [cutAB4] using hi
    have h4 : i.val < 4 := i.isLt
    by_cases h2 : i.val = 2
    · simp only [echange, if_pos h2]
      exact heq 3 rfl
    · have h3 : i.val = 3 := by omega
      simp only [echange, if_neg h2, if_pos h3]
      exact heq 2 rfl


-- Le verdict, par le theoreme A : chaque cote ne lit que lui-meme.
theorem echange_decomposable : decomposable echange := by
  refine product_decomposable cutAB4 echange ?_ ?_
  · -- Cote A : deux etats qui coïncident sur {0,1} ont des images qui y coïncident.
    intro s s' heq i hi
    have hi2 : i.val < 2 := by simpa [cutAB4] using hi
    by_cases h2 : i.val = 2
    · exact absurd h2 (by omega)
    · have h3 : i.val ≠ 3 := by omega
      simp only [echange, if_neg h2, if_neg h3]
      exact heq i hi
  · -- Cote B : deux etats qui coïncident sur {2,3} ont des images qui y coïncident.
    intro s s' heq i hi
    have hlt : ¬ (i.val < 2) := by simpa [cutAB4] using hi
    have h4 : i.val < 4 := i.isLt
    by_cases h2 : i.val = 2
    · simp only [echange, if_pos h2]
      exact heq 3 rfl
    · have h3 : i.val = 3 := by omega
      simp only [echange, if_neg h2, if_pos h3]
      exact heq 2 rfl

--% env 6

Raw input:
{"cmd": "-- Le verdict, par le theoreme A : chaque cote ne lit que lui-meme.\ntheorem echange_decomposable : decomposable echange := by\n  refine product_decomposable cutAB4 echange ?_ ?_\n  \u00b7 -- Cote A : deux etats qui co\u00efncident sur {0,1} ont des images qui y co\u00efncident.\n    intro s s' heq i hi\n    have hi2 : i.val < 2 := by simpa [cutAB4] using hi\n    by_cases h2 : i.val = 2\n    \u00b7 exact absurd h2 (by omega)\n    \u00b7 have h3 : i.val \u2260 3 := by omega\n      simp only [echange, if_neg h2, if_neg h3]\n      exact heq i hi\n  \u00b7 -- Cote B : deux etats qui co\u00efncident sur {2,3} ont des images qui y co\u00efncident.\n    intro s s' heq i hi\n    have hlt : \u00ac (i.val < 2) := by simpa [cutAB4] using hi\n    have h4 : i.val < 4 := i.isLt\n    by_cases h2 : i.val = 2\n    \u00b7 simp only [echange, if_pos h2]\n      exact heq 3 rfl\n    \u00b7 have h3 : i.val = 3 := by omega\n      simp only [echange, if_neg h2, if_pos h3]\n      exact heq 2 rfl\n", "env": 5}
Raw output:
{"env": 6}

**Interprétation.** Le verdict est tombé : `decomposable echange`, **preuve à l'appui** —
et nous n'avons **pas énuméré les coupes** ni les états. Le théorème a fait le travail : les deux
hypothèses (`hA`, `hB`) sont de simples lectures de la définition de `echange` — côté A, les bits
0-1 traversent le `if` sans être touchés (`if_neg` × 2) ; côté B, l'image d'un bit ne lit que
l'autre bit côté B (`heq 3 rfl`, `heq 2 rfl`). Notez la symétrie des deux branches : la
non-trivialité du côté B (l'échange) n'a **coûté aucune ligne de plus** — l'autonomie, pas
l'inertie, est ce que le théorème exige.


## 6. La même machinerie bute sur le broadcast

Appliquons maintenant l'instrument au système central de l'Epic : le **broadcast-parité**, où
l'état suivant est uniformément égal à la parité de l'état courant. La marche 2 lui consacrera sa
construction ; ici, nous n'avons besoin que de son comportement sur les **motifs**.


In [8]:
#check @IIT.parity
#check @IIT.broadcast

-- Les huit etats de n = 3, en ordre binaire croissant (000, 001, ..., 111).
def huitEtats : List (State 3) :=
  [ fun _ => false,
    fun i => i.val == 2,
    fun i => i.val == 1,
    fun i => 1 ≤ i.val,
    fun i => i.val == 0,
    fun i => i.val ≠ 1,
    fun i => i.val ≠ 2,
    fun _ => true ]

-- L'image du broadcast ne depend que de la parite.
#eval huitEtats.map (fun s => parity 3 s)


#check @IIT.parity
──────▶  parity : (n : Nat) → State n → Bool
#check @IIT.broadcast
──────▶  @broadcast : {n : Nat} → System n

-- Les huit etats de n = 3, en ordre binaire croissant (000, 001, ..., 111).
def huitEtats : List (State 3) :=
  [ fun _ => false,
    fun i => i.val == 2,
    fun i => i.val == 1,
    fun i => 1 ≤ i.val,
    fun i => i.val == 0,
    fun i => i.val ≠ 1,
    fun i => i.val ≠ 2,
    fun _ => true ]

-- L'image du broadcast ne depend que de la parite.
#eval huitEtats.map (fun s => parity 3 s)
─────▶  [false, true, true, false, true, false, false, true]

--% env 7

Raw input:
{"cmd": "#check @IIT.parity\n#check @IIT.broadcast\n\n-- Les huit etats de n = 3, en ordre binaire croissant (000, 001, ..., 111).\ndef huitEtats : List (State 3) :=\n  [ fun _ => false,\n    fun i => i.val == 2,\n    fun i => i.val == 1,\n    fun i => 1 \u2264 i.val,\n    fun i => i.val == 0,\n    fun i => i.val \u2260 1,\n    fun i => i.val \u2260 2,\n    fun _ => true ]\n\n-- L'image du broadcast ne depend que de la parite.\n#eval huitEtats.map (fun s => parity 3 s)\n", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "parity : (n : Nat) → State n → Bool"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "@broadcast : {n : Nat} → System n"},
  {"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 5},
   "data": "[false, true, true, false, true, false, false, true]"}],
 "env": 7}

**Interprétation.** Huit états, deux images : `false, true, true, false, true, false, false, true`.
Tout le contenu informationnel de l'état est réduit à **un bit** — la parité — puis diffusé
partout. Énumérons maintenant la table des **couples de motifs** pour la coupe minimale
A = {0} : pour chaque état `s`, le triplet `[motif A de l'image, motif B de l'image (bit 1), motif B (bit 2)]`.


In [9]:
-- Coupe minimale : A = {0}, B = {1, 2}.
def coupeMin : Cut 3 where
  sel := fun i => i.val == 0
  ha := ⟨0, rfl⟩
  hb := ⟨1, rfl⟩

-- Pour chaque etat : le motif A de l'image, puis les deux bits du motif B.
#eval huitEtats.map (fun s =>
  [ projT coupeMin (broadcast s) 0,
    projF coupeMin (broadcast s) 1,
    projF coupeMin (broadcast s) 2 ])


-- Coupe minimale : A = {0}, B = {1, 2}.
def coupeMin : Cut 3 where
  sel := fun i => i.val == 0
  ha := ⟨0, rfl⟩
  hb := ⟨1, rfl⟩

-- Pour chaque etat : le motif A de l'image, puis les deux bits du motif B.
#eval huitEtats.map (fun s =>
─────▶  [[false, false, false],
 [true, true, true],
 [true, true, true],
 [false, false, false],
 [true, true, true],
 [false, false, false],
 [false, false, false],
 [true, true, true]]
  [ projT coupeMin (broadcast s) 0,
    projF coupeMin (broadcast s) 1,
    projF coupeMin (broadcast s) 2 ])

--% env 8

Raw input:
{"cmd": "-- Coupe minimale : A = {0}, B = {1, 2}.\ndef coupeMin : Cut 3 where\n  sel := fun i => i.val == 0\n  ha := \u27e80, rfl\u27e9\n  hb := \u27e81, rfl\u27e9\n\n-- Pour chaque etat : le motif A de l'image, puis les deux bits du motif B.\n#eval huitEtats.map (fun s =>\n  [ projT coupeMin (broadcast s) 0,\n    projF coupeMin (broadcast s) 1,\n    projF coupeMin (broadcast s) 2 ])\n", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data":
   "[[false, false, false],\n [true, true, true],\n [true, true, true],\n [false, false, false],\n [true, true, true],\n [false, false, false],\n [false, false, false],\n [true, true, true]]"}],
 "env": 8}

**Interprétation.** Chaque ligne est uniforme — `[false, false, false]` ou `[true, true, true]`.
Le **couple** (motif A = `[true]`, motif B = `[false, false]`) n'apparaît **jamais**, alors que
chaque motif séparément est réalisable : `projT` vaut `[true]` sur les quatre états impairs,
`projF` vaut `[false, false]` sur les états pairs. C'est exactement le couple que le témoin du
théorème A devrait réaliser et **ne peut pas** : il faudrait un état `z` avec `parity 3 z = true`
**et** `parity 3 z = false`. La machinerie de la §5 bute ici — et c'est un théorème, pas une
coïncidence sur `n = 3` : `broadcast_integrated` (marche 3) établira cette butée pour **toute**
coupe et **tout** `n`.

Dernière lecture, la plus importante : le contraste des §5-§6 ne dit **rien** du comportement des
systèmes. `echange` est décomposable et son orbite vit sur place ; le broadcast est intégré au
sens du proxy et sa dynamique atteint un point fixe en deux pas (marche 3). Verdict de coupe et
richesse comportementale sont des axes **orthogonaux** — c'est tout le contenu de la critique
d'Aaronson.


## 7. Ce que le verdict ne dit pas — le grade du proxy

Le lake le porte lui-même dans sa documentation, et ce carnet le répète parce que c'est la
condition de lecture honnête :

| Le proxy `cutIndep` | Φ (la théorie) |
|---|---|
| verdict **binaire** : une coupe sépare ou ne sépare pas | quantité réelle, minimisée sur les partitions (MIP) |
| décomposition **déterministe de l'image** | information **effective sous perturbation** |
| aucune valeur numérique | $\Phi$ en bits, comparé entre systèmes |
| énoncé sur `System n : Fin n → Bool` | défini sur TPM probabilistes |

Ce que la marche 1 établit n'est donc **pas** « le broadcast est conscient » ni même « intégré au
sens de Tononi » : c'est « le broadcast n'admet **aucune** coupe qui réalise le produit de ses
parties, au sens exact et constructif de `cutIndep` ». La mesure quantitative — la cardinalité
`|imA| · |imB| - |im f|` — est renvoyée à une phase ultérieure du lake. Pour la Φ_DM et les grilles
PyPhi côté empirique : [IIT-6](./IIT-06-L-Objet-qui-a-Mordu-IIT-Python.ipynb).


## 8. Exercices

Trois exercices bornés pour manipuler les objets de la marche 1. Chaque cellule **s'exécute telle
quelle** (corps trivial) : à vous de remplacer le corps par la bonne définition. Les `#eval` de
contrôle affichent côte à côte votre valeur et celle du lake — la divergence signale l'exercice
inachevé, pas une erreur du notebook.


### Exercice 1 — écrire une coupe

Définissez `selPairs` plaçant côté A exactement les éléments d'**indice pair** de `n = 4`
(attendu : `[true, false, true, false]`). Le corps actuel rend `false` partout.


In [10]:
-- Exercice 1 : true si et seulement si l'indice est pair.
-- TODO etudiant : remplacer le corps ci-dessous
def selPairs (_ : Fin 4) : Bool :=
  false

-- Doit valoir [true, false, true, false]
#eval [selPairs 0, selPairs 1, selPairs 2, selPairs 3]


-- Exercice 1 : true si et seulement si l'indice est pair.
-- TODO etudiant : remplacer le corps ci-dessous
def selPairs (_ : Fin 4) : Bool :=
  false

-- Doit valoir [true, false, true, false]
#eval [selPairs 0, selPairs 1, selPairs 2, selPairs 3]
─────▶  [false, false, false, false]

--% env 9

Raw input:
{"cmd": "-- Exercice 1 : true si et seulement si l'indice est pair.\n-- TODO etudiant : remplacer le corps ci-dessous\ndef selPairs (_ : Fin 4) : Bool :=\n  false\n\n-- Doit valoir [true, false, true, false]\n#eval [selPairs 0, selPairs 1, selPairs 2, selPairs 3]\n", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "[false, false, false, false]"}],
 "env": 9}

### Exercice 2 — le motif d'un côté

Définissez `motifA` rendant la liste des quatre composantes du motif côté A (`projT`) d'un état
pour une coupe — sans appeler `projT` dans un `#eval` : construisez la liste à la main, composante
par composante, avec des `if`. Le contrôle croise avec `projT` du lake sur l'état et la coupe de
la §5 : attendu `[true, true, false, false]`.


In [11]:
-- Exercice 2 : les quatre composantes du motif cote A, reconstruites a la main.
-- TODO etudiant : remplacer le corps ci-dessous
def motifA (c : Cut 4) (s : State 4) : List Bool :=
  [false, false, false, false]

-- Doit valoir [true, true, false, false]
#eval motifA cutAB4 etatTemoin
#eval [projT cutAB4 etatTemoin 0, projT cutAB4 etatTemoin 1, projT cutAB4 etatTemoin 2, projT cutAB4 etatTemoin 3]


-- Exercice 2 : les quatre composantes du motif cote A, reconstruites a la main.
-- TODO etudiant : remplacer le corps ci-dessous
def motifA (c : Cut 4) (s : State 4) : List Bool :=
            ─▶ 🟨 Variable name `c` is not explicitly referenced.

Hint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:
  [apply] _c

Note: This linter can be disabled with `set_option linter.unusedVariables false`
                        ─▶ 🟨 Variable name `s` is not explicitly referenced.

Hint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:
  [apply] _s

Note: This linter can be disabled with `set_option linter.unusedVariables false`
  [false, false, false, false]

-- Doit valoir [true, true, false, false]
#eval motifA cutAB4 etatTemoin
─────▶  [false, false, false, false]
#eval [projT cutAB4 etatTemoin 0, projT cutAB4 etatTemoin 1, projT cutAB4 etatTemoin 2, projT cutAB4 etatTemoin 3]
─────▶  [true, false, false, false]

--% env 10

Raw input:
{"cmd": "-- Exercice 2 : les quatre composantes du motif cote A, reconstruites a la main.\n-- TODO etudiant : remplacer le corps ci-dessous\ndef motifA (c : Cut 4) (s : State 4) : List Bool :=\n  [false, false, false, false]\n\n-- Doit valoir [true, true, false, false]\n#eval motifA cutAB4 etatTemoin\n#eval [projT cutAB4 etatTemoin 0, projT cutAB4 etatTemoin 1, projT cutAB4 etatTemoin 2, projT cutAB4 etatTemoin 3]\n", "env": 9}
Raw output:
{"messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 12},
   "endPos": {"line": 3, "column": 13},
   "data":
   "Variable name `c` is not explicitly referenced.\n\nHint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:\n  [apply] _c\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"},
  {"severity": "warning",
   "pos": {"line": 3, "column": 24},
   "endPos": {"line": 3, "column": 25},
   "data":
   "Variable name `s` is not explicitly referenced.\n\nHint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:\n  [apply] _s\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "[false, false, false, false]"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "[true, false, false, false]"}],
 "env": 10}

### Exercice 3 — reconstruire la parité

Reconstruisez la parité d'un état de `n = 4` éléments **sans** appeler `parity` du lake : avec des
`xor` sur les composantes. Le contrôle croise sur trois états : tout-faux (attendu `false`),
one-hot en 0 (attendu `true`), et deux bits vrais (attendu `false`).


In [12]:
-- Exercice 3 : la parite d'un etat de n = 4, reconstruite avec xor.
-- TODO etudiant : remplacer le corps ci-dessous
def pariteA4 (s : State 4) : Bool :=
  false

-- Chaque paire doit rendre la meme valeur
#eval pariteA4 (fun _ => false)
#eval parity 4 (fun _ => false)
#eval pariteA4 (fun i => i.val == 0)
#eval parity 4 (fun i => i.val == 0)
#eval pariteA4 (fun i => i.val < 2)
#eval parity 4 (fun i => i.val < 2)


-- Exercice 3 : la parite d'un etat de n = 4, reconstruite avec xor.
-- TODO etudiant : remplacer le corps ci-dessous
def pariteA4 (s : State 4) : Bool :=
              ─▶ 🟨 Variable name `s` is not explicitly referenced.

Hint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:
  [apply] _s

Note: This linter can be disabled with `set_option linter.unusedVariables false`
  false

-- Chaque paire doit rendre la meme valeur
#eval pariteA4 (fun _ => false)
─────▶  false
#eval parity 4 (fun _ => false)
─────▶  false
#eval pariteA4 (fun i => i.val == 0)
─────▶  false
#eval parity 4 (fun i => i.val == 0)
─────▶  true
#eval pariteA4 (fun i => i.val < 2)
─────▶  false
#eval parity 4 (fun i => i.val < 2)
─────▶  false

--% env 11

Raw input:
{"cmd": "-- Exercice 3 : la parite d'un etat de n = 4, reconstruite avec xor.\n-- TODO etudiant : remplacer le corps ci-dessous\ndef pariteA4 (s : State 4) : Bool :=\n  false\n\n-- Chaque paire doit rendre la meme valeur\n#eval pariteA4 (fun _ => false)\n#eval parity 4 (fun _ => false)\n#eval pariteA4 (fun i => i.val == 0)\n#eval parity 4 (fun i => i.val == 0)\n#eval pariteA4 (fun i => i.val < 2)\n#eval parity 4 (fun i => i.val < 2)\n", "env": 10}
Raw output:
{"messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 14},
   "endPos": {"line": 3, "column": 15},
   "data":
   "Variable name `s` is not explicitly referenced.\n\nHint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:\n  [apply] _s\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "false"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "false"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 5},
   "data": "false"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 5},
   "data": "true"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "false"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 5},
   "data": "false"}],
 "env": 11}

## 9. Conclusion — ce que la marche 1 a établi

| Geste | Outil du lake | Résultat |
|---|---|---|
| Tenir un système et une coupe comme objets Lean | `State`, `System`, `Cut` | états, images et motifs **calculés** sur des instances |
| Lire l'indépendance comme un produit d'images | `projT`, `projF`, `cutIndep` | le verdict est **relationnel**, binaire, constructif |
| Faire rendre un verdict décomposable | `cutIndep_of_sides_independent` → `product_decomposable` | `echange_decomposable` prouvé **sans énumération**, hypothèses lues sur la définition |
| Constater la butée du broadcast | `broadcast`, `parity` | le couple de motifs `(A=true, B=false)` réalisable séparément, **jamais ensemble** — table calculée sur les 8 états |

La marche 2 calculera la parité de bout en bout (`parity`, ses quatre lemmes) ; la marche 3
énoncera la butée comme théorème universel (`broadcast_integrated`) et la dyna­mique triviale
(`broadcast_two_cycle`) — le couple qui **est** la critique d'Aaronson sous forme formelle. Le
pont avec la moitié empirique (Φ_DM sur F_p, grilles PyPhi) se refera là.

**Ce que ce carnet ne claim pas** : aucune valeur de Φ, aucune conscience mesurée — un proxy
relationnel de décomposition, exactement ce que le lake formalise.
